# SOHO matched-selection three-dataset study
This V2 notebook compares **SOHO replay**, **official FLY**, **validation-tuned FLY**, and **raw Ridge**. SOHO, tuned FLY and raw Ridge select their own hyperparameters on identical train-only nested splits. Official FLY remains a separate fidelity control. SOHO replay is not exemplar-free.

In [ ]:
# === Edit repository/path values only. Do not edit protocol seeds or search spaces. ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'experiment/soho-selfcontained'
WORK_DIR = '/content/SOHO-CL'
FEATURE_CACHE_ROOT = '/content/soho_matched_features'
SELECTION_ROOT = '/content/soho_matched_selection'
OUTPUT_ROOT = '/content/soho_matched_results'
BATCH_SIZE = 128
NUM_WORKERS = 2
EXPECTED_PROTOCOL_SHA256 = 'ab3e7fde1d375a96231ac26508926c21cbe74eee2474b1743b15589916694aeb'
EXPECTED_RUNNER_SHA256 = '4805a2192839314bb3710432e1e6aa4019b53d37cf798a0594957dc201e9ec97'
EXPECTED_BASE_RUNNER_SHA256 = '5d3ee38b93a10f4fdfa52aaeb1b901df97b0d40e5c806b67a2d609da094234fc'

In [ ]:
# Fresh clone, dependencies, GPU check and immutable source verification.
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
os.chdir('/content')
repo=Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR],check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub','pandas','matplotlib','seaborn'],check=True)
import torch
assert torch.cuda.is_available(), 'Select Runtime -> Change runtime type -> T4 GPU.'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
PROTOCOL='configs/soho_matched_selection_final.json'
RUNNER='tools/soho_matched_selection.py'
BASE_RUNNER='tools/soho_selfcontained.py'
assert sha(PROTOCOL)==EXPECTED_PROTOCOL_SHA256
assert sha(RUNNER)==EXPECTED_RUNNER_SHA256
assert sha(BASE_RUNNER)==EXPECTED_BASE_RUNNER_SHA256
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip()
print('GPU:',torch.cuda.get_device_name(0))
print('commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('MATCHED V2 SOURCE CHECK: PASS')

In [ ]:
# Download the verified frozen ViT checkpoint and processed datasets.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==346284714
assert sha(CHECKPOINT_PATH)=='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
DATASET_ROOTS={'cifar100':kagglehub.dataset_download('zaphat206/cifar-100'),'cub200':kagglehub.dataset_download('zaphat206/cub-200-2011'),'imagenetr':kagglehub.dataset_download('zaphat206/imagenet-r')}
print(json.dumps(DATASET_ROOTS,indent=2))

In [ ]:
# Audit dataset identities without extracting test features.
CUB_AUDIT='/content/cub_soho_matched_audit.json'
IMAGENETR_AUDIT='/content/imagenetr_soho_matched_audit.json'
assert subprocess.run([sys.executable,'-u','tools/cub_dataset_audit.py','--root',DATASET_ROOTS['cub200'],'--output',CUB_AUDIT,'--expected-identity-sha256','e374af9b576cb6b3503198ef3ea30fd0aa9d2e18c230ff8064e21d4f644af2ca']).returncode==0
assert subprocess.run([sys.executable,'-u','tools/imagenetr_dataset_audit.py','--root',DATASET_ROOTS['imagenetr'],'--output',IMAGENETR_AUDIT,'--expected-identity-sha256','3f3d963b2b0c245ceabc0166c8b1c64d624c2ea31df07ee6ffdbf4cab5f7739d','--diagnose-cross-split-duplicates','--workers','4']).returncode==2
audit=json.loads(Path(IMAGENETR_AUDIT).read_text())
assert audit['cross_split_duplicate_content_count']==19 and audit['cross_split_conflicting_label_duplicate_count']==18
print('DATASET AUDIT PASS; ImageNet-R remains a legacy processed split.')

In [ ]:
# Extract frozen TRAIN features only. No Drive storage and no visible test.pt.
protocol=json.loads(Path(PROTOCOL).read_text())
Path(FEATURE_CACHE_ROOT).mkdir(parents=True,exist_ok=True)
for key in ('cifar100','cub200','imagenetr'):
    cfg=protocol['datasets'][key]; cache=Path(FEATURE_CACHE_ROOT)/key
    if not (cache/'train.pt').is_file():
        command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',DATASET_ROOTS[key],'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256',protocol['backbone']['checkpoint_sha256'],'--feature-cache-dir',str(cache),'--output-dir',f'/content/unused_matched_{key}','--dataset',cfg['dataset'],'--model-name',protocol['backbone']['model_name'],'--data-augmentation','vit','--seed','2025','--num-classes',str(cfg['num_classes']),'--num-tasks',str(cfg['num_tasks']),'--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
        print(f'TRAIN EXTRACT START {key}',flush=True); subprocess.run(command,check=True)
    assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
    print(f'TRAIN CACHE READY {key}; test.pt absent')
print('ALL TRAIN-ONLY FEATURE CACHES READY')

In [ ]:
# Synthetic correctness/fidelity gate with complete output.
command=[sys.executable,'-B','-m','pytest','-q','tests/test_soho_matched_selection.py','tests/test_soho_selfcontained.py','tests/test_cached_replay_baselines.py']
completed=subprocess.run(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
print(completed.stdout,flush=True)
if completed.returncode!=0: raise RuntimeError(f'Correctness gate failed: {completed.returncode}')
print('SOHO MATCHED-SELECTION CORRECTNESS GATE: PASS')

In [ ]:
# TRAIN-ONLY nested selection: SOHO, raw Ridge and 18 FLY candidates use identical splits.
audit_paths={'cifar100':None,'cub200':CUB_AUDIT,'imagenetr':IMAGENETR_AUDIT}
for key in ('cifar100','cub200','imagenetr'):
    command=[sys.executable,'-u',RUNNER,'select','--protocol',PROTOCOL,'--dataset-key',key,'--feature-cache-dir',str(Path(FEATURE_CACHE_ROOT)/key),'--output-root',SELECTION_ROOT,'--device','cuda']
    if audit_paths[key]: command += ['--dataset-audit',audit_paths[key]]
    print(f'MATCHED SELECTION START {key}: SOHO + raw Ridge + 18 FLY configs x 3 replicates',flush=True)
    subprocess.run(command,check=True)
    selected=json.loads(Path(SELECTION_ROOT,key,'selection.json').read_text())
    print(key,'SOHO=',selected['selected_soho_config'],'FLY tuned=',selected['selected_fly_config'],'raw lambda=',selected['selected_raw_ridge_lambda'])
    assert selected['status']=='SELECTION_COMPLETE' and selected['uses_test_set'] is False
print('MATCHED TRAIN-ONLY SELECTION COMPLETE')

In [ ]:
# Inspect validation evidence and create the immutable authorization lock.
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
rows=[]; fly_rows=[]
for key in ('cifar100','cub200','imagenetr'):
    result=json.loads(Path(SELECTION_ROOT,key,'selection.json').read_text())
    rows.append({'dataset':key,**result['selected_soho_config'],'raw_ridge_lambda':result['selected_raw_ridge_lambda'],**{f'fly_{k}':v for k,v in result['selected_fly_config'].items()}})
    for candidate in result['fly_candidates']:
        for ridx,replicate in enumerate(candidate['per_replicate']): fly_rows.append({'dataset':key,'replicate':ridx,**candidate['config'],'validation_AIA':replicate['average_incremental_accuracy']})
display(pd.DataFrame(rows))
fly_df=pd.DataFrame(fly_rows); plot_dir=Path(OUTPUT_ROOT)/'validation_plots'; plot_dir.mkdir(parents=True,exist_ok=True)
fig,axes=plt.subplots(1,3,figsize=(22,6))
for ax,key in zip(axes,('cifar100','cub200','imagenetr')):
    table=fly_df[fly_df.dataset==key].groupby(['synaptic_degree','coding_level']).validation_AIA.mean().unstack()
    sns.heatmap(table,annot=True,fmt='.2f',cmap='viridis',ax=ax); ax.set_title(f'{key}: tuned FLY validation AIA')
fig.tight_layout(); fig.savefig(plot_dir/'fly_matched_grid.png',dpi=220,bbox_inches='tight'); plt.show()
dirty=subprocess.check_output(['git','status','--porcelain'],text=True).strip(); assert not dirty,f'Repository changed before lock:\n{dirty}'
Path(OUTPUT_ROOT).mkdir(parents=True,exist_ok=True)
subprocess.run([sys.executable,'-u',RUNNER,'lock','--protocol',PROTOCOL,'--selection-root',SELECTION_ROOT,'--output-root',OUTPUT_ROOT,'--require-clean-git'],check=True)
AUTHORIZATION=str(Path(OUTPUT_ROOT)/'authorization.json')
print(json.dumps(json.loads(Path(AUTHORIZATION).read_text()),indent=2))

## Authorized test boundary
All method-specific search spaces, selections, source hashes and seeds are immutable. Do not edit anything after viewing test output. These test splits were used in earlier repository phases, so this is a locked comparative evaluation rather than first-use untouched held-out evidence.

In [ ]:
# Materialize TEST features only after authorization; task-loader iteration is fixed in V2.
for key in ('cifar100','cub200','imagenetr'):
    command=[sys.executable,'-u',RUNNER,'extract-test','--protocol',PROTOCOL,'--dataset-key',key,'--selection-root',SELECTION_ROOT,'--authorization',AUTHORIZATION,'--feature-cache-dir',str(Path(FEATURE_CACHE_ROOT)/key),'--root',DATASET_ROOTS[key],'--backbone-checkpoint',CHECKPOINT_PATH,'--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print(f'TEST EXTRACTION START {key}',flush=True); subprocess.run(command,check=True)
print('ALL AUTHORIZED TEST FEATURE CACHES READY')

In [ ]:
# Helper: refit from empty state on full train, then run six paired final replicates.
def run_final(key):
    command=[sys.executable,'-u',RUNNER,'evaluate','--protocol',PROTOCOL,'--dataset-key',key,'--selection-root',SELECTION_ROOT,'--authorization',AUTHORIZATION,'--feature-cache-dir',str(Path(FEATURE_CACHE_ROOT)/key),'--output-root',OUTPUT_ROOT,'--device','cuda']
    if audit_paths[key]: command += ['--dataset-audit',audit_paths[key]]
    print(f'FINAL START {key}: 6 replicates x 4 methods',flush=True); subprocess.run(command,check=True)
    payload=json.loads(Path(OUTPUT_ROOT,key,'final_results.json').read_text()); rows=[]
    for replicate in payload['seed_results']:
        for method,result in replicate['methods'].items(): rows.append({'replicate':replicate['replicate_index'],'method':method,'final_accuracy':result.get('final_accuracy'),'AIA':result.get('average_incremental_accuracy'),'state_MiB':None if result.get('persistent_state_bytes') is None else result['persistent_state_bytes']/2**20,'exemplar_free':result.get('state_audit',{}).get('exemplar_free')})
    display(pd.DataFrame(rows)); print('STATUS:',payload['status'])

In [ ]:
# Run separately so completed datasets survive interruption.
run_final('cifar100')

In [ ]:
run_final('cub200')

In [ ]:
run_final('imagenetr')

In [ ]:
# Aggregate, paired comparisons and publication-oriented plots.
subprocess.run([sys.executable,'-u',RUNNER,'summarize','--protocol',PROTOCOL,'--output-root',OUTPUT_ROOT],check=True)
metrics=pd.read_csv(Path(OUTPUT_ROOT)/'metrics_summary.csv'); display(metrics.sort_values(['dataset','method']))
summary=json.loads(Path(OUTPUT_ROOT,'final_summary.json').read_text()); print(json.dumps(summary['paired_aia_differences'],indent=2))
curves=pd.read_csv(Path(OUTPUT_ROOT)/'task_curves.csv'); names={'soho_replay_fidelity':'SOHO replay','flycl_fidelity':'FLY official','flycl_validation_tuned':'FLY tuned','raw_ridge':'Raw Ridge'}
fig,axes=plt.subplots(1,3,figsize=(21,5.5),sharey=True)
for ax,key in zip(axes,('cifar100','cub200','imagenetr')):
    view=curves[curves.dataset==key]
    for method,label in names.items():
        mean=view[view.method==method].groupby('task').average_seen_accuracy.mean(); ax.plot(mean.index,mean.values,label=label,marker='o',ms=3)
    ax.set_title(key); ax.set_xlabel('Task'); ax.set_ylabel('Average seen-class accuracy (%)')
axes[0].legend(fontsize=9); fig.tight_layout(); fig.savefig(Path(OUTPUT_ROOT)/'matched_accuracy_curves.png',dpi=220,bbox_inches='tight'); plt.show()
fig,axes=plt.subplots(1,3,figsize=(18,5.5),sharey=True)
for ax,key in zip(axes,('cifar100','cub200','imagenetr')):
    view=metrics[metrics.dataset==key]
    for method,label in names.items():
        row=view[view.method==method].iloc[0]; ax.scatter(row.persistent_state_bytes_mean/2**20,row.average_incremental_accuracy_mean,s=110); ax.annotate(label,(row.persistent_state_bytes_mean/2**20,row.average_incremental_accuracy_mean),fontsize=8)
    ax.set_xscale('log'); ax.set_title(key); ax.set_xlabel('Persistent state (MiB, log)'); ax.set_ylabel('AIA (%)')
fig.tight_layout(); fig.savefig(Path(OUTPUT_ROOT)/'matched_accuracy_memory.png',dpi=220,bbox_inches='tight'); plt.show()

In [ ]:
# Export compact evidence; feature caches and SOHO replay tensors are excluded.
from google.colab import files
shutil.copy2(PROTOCOL,Path(OUTPUT_ROOT)/'locked_protocol.json'); shutil.copy2(RUNNER,Path(OUTPUT_ROOT)/'locked_runner.py'); shutil.copy2(BASE_RUNNER,Path(OUTPUT_ROOT)/'locked_base_runner.py')
selection_copy=Path(OUTPUT_ROOT)/'train_only_selection'; selection_copy.mkdir(exist_ok=True)
for key in ('cifar100','cub200','imagenetr'): shutil.copy2(Path(SELECTION_ROOT,key,'selection.json'),selection_copy/f'{key}_selection.json')
archive=shutil.make_archive('/content/soho_matched_selection_three_dataset_results','zip',root_dir=OUTPUT_ROOT)
print('artifact:',archive,'size=',Path(archive).stat().st_size,'SHA-256=',sha(archive)); files.download(archive)